In [91]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer # Text -> Numeric
from sklearn.cluster import KMeans
from umap import UMAP # Dimensionality Reduction for Visualization
import plotly.express as px  # Interactive Visualization

In [92]:
# 1. Gather review/text data
kol_posts = pd.read_csv("../data/kol/KOL_Posts.csv")
rest_reviews = pd.read_parquet("../_3_marketing/restaurant_reviews.parquet")
rest_best_reviews = pd.read_parquet("../_3_marketing/restaurant_reviews_all_best.parquet")
rest = pd.read_parquet("../data/mv_dataset_parquet/restaurants.parquet")
placesapinew = pd.read_csv("../_1_eda/places_api_new_results.csv")

#rest[rest['name'] == 'Melody Bangkok']
#kol_posts[kol_posts['Restaurant Name'] == 'Audrey Cafe Thonglor Soi 11']
#placesapinew[placesapinew['name'] == 'JW Cafe at JW Marriott Hotel Bangkok']

In [93]:
# Aggregate KOL Posts per restaurant
kol_aggregated = (
    kol_posts.groupby('Restaurant Name')['Content']
    .apply(lambda x: ' '.join(x.dropna().astype(str)))
    .reset_index()
)
kol_aggregated.columns = ['Restaurant Name', 'kol_text']

#print(kol_aggregated)

placesapinew['google_text'] = ( # Combine relevant fields from placesapinew into one text string for each restaurant
        placesapinew['input_string'].fillna('') + ' ' +
        placesapinew['raw_types'].fillna('').str.replace(',', ' ') + ' ' +
        placesapinew['Cuisine'].fillna('')
)

# Merge KOL aggregated text with placesapinew to get a combined dataset for clustering
kol_restaurant_mapping = kol_posts[['Restaurant Name', 'Restaurant Code']].drop_duplicates()
places_merged = placesapinew.merge(
    kol_restaurant_mapping,
    left_on='input_string', # Assuming 'input_string' in placesapinew corresponds to 'Restaurant Name' in kol_posts
    right_on='Restaurant Name', # Merge on restaurant name
    how='left'
)

# Merge dataset with both the google text and the KOL text for each restaurant
places_aggregated = places_merged[['Restaurant Name', 'google_text']].rename(
    columns={'Restaurant Name': 'restaurant name'}
)

rest_reviews['rest_reviews'] = ( # Combine restaurant name and review text into one string for clustering
    rest_reviews['input_restaurant_name'].fillna('') + ' ' +
    rest_reviews['review_text'].fillna('').str.replace(',', ' ') 
)

rest_places_merged = rest_reviews.merge( # Merge restaurant reviews with the aggregated places data to get a combined dataset for clustering
    places_aggregated,
    left_on='input_restaurant_name', # Assuming 'input_string' in rest_reviews corresponds to 'restaurant name' in places_aggregated
    right_on='restaurant name', # Merge on restaurant name
    how='left'
)

places_aggregated.head()
rest_places_merged.head()

,input_restaurant_name,pulled_at_utc,source,place_id,matched_name,review_id,author_name,author_url,profile_photo_url,rating,language,review_text,relative_time_description,review_time_unix,review_time_utc,rest_reviews,restaurant name,google_text
0,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1770211654_NATTAWA...,NATTAWAT RATTANAPONGPRAKIT,https://www.google.com/maps/contrib/1024293807...,https://lh3.googleusercontent.com/a-/ALV-UjVOX...,5,en-US,We ate it all before I could take a picture! T...,a week ago,1770211654,2026-02-04T13:27:34+00:00,Melody Bangkok We ate it all before I could ta...,NaN,NaN
1,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1770021873_Mizutan...,Mizutani Koichi,https://www.google.com/maps/contrib/1102947333...,https://lh3.googleusercontent.com/a-/ALV-UjU8O...,1,en-US,The Happy Hour menu does not include draft bee...,2 weeks ago,1770021873,2026-02-02T08:44:33+00:00,Melody Bangkok The Happy Hour menu does not in...,NaN,NaN
2,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769797681_Patpoom...,Patpoom MSoulScent,https://www.google.com/maps/contrib/1168736595...,https://lh3.googleusercontent.com/a-/ALV-UjVEH...,5,en-US,"The staff are nice, friendly, and polite.\nThe...",2 weeks ago,1769797681,2026-01-30T18:28:01+00:00,Melody Bangkok The staff are nice friendly a...,NaN,NaN
3,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769692334_Jesada ...,Jesada Bobpahow,https://www.google.com/maps/contrib/1041664502...,https://lh3.googleusercontent.com/a-/ALV-UjVw9...,5,en-US,"Delicious food, great atmosphere.",2 weeks ago,1769692334,2026-01-29T13:12:14+00:00,Melody Bangkok Delicious food great atmosphere.,NaN,NaN
4,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769692169_อรทัย ห...,อรทัย หลงทัพไทย,https://www.google.com/maps/contrib/1091362972...,https://lh3.googleusercontent.com/a/ACg8ocLqF_...,5,en-US,"The taste is good, the food is delicious.",2 weeks ago,1769692169,2026-01-29T13:09:29+00:00,Melody Bangkok The taste is good the food is ...,NaN,NaN


In [94]:
# 2. Prepare text data

rest_places_aggregated = ( # Aggregate reviews and google text by restaurant name to get one combined text string per restaurant for clustering
    rest_places_merged
    .groupby('input_restaurant_name')[['rest_reviews', 'google_text']]  # use double brackets
    .agg({
        'rest_reviews': lambda x: ' '.join(x.dropna().astype(str)),
        'google_text': lambda x: ' '.join(x.dropna().astype(str))
    })
    .reset_index()
    .rename(columns={'input_restaurant_name': 'restaurant name'})
)

# Combine both rest reviews and google text columns into one
rest_places_aggregated['text'] = (
    rest_places_aggregated['rest_reviews'].fillna('') + ' ' +
    rest_places_aggregated['google_text'].fillna('')
)

reviews = rest_places_aggregated[['restaurant name', 'text']].copy() # Final dataset for clustering with restaurant name and combined text

reviews

,restaurant name,text
0,&T modern Thai grill,&T modern Thai grill -The restaurant is beauti...
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,'@RICE Restaurant by At Rice Resort (Nakhon Na...
2,100 Degrees Hotpot Central Chaengwattana,100 Degrees Hotpot Central Chaengwattana Delic...
3,100 Degrees Hotpot Cosmo Bazaar,100 Degrees Hotpot Cosmo Bazaar Kimchi soup is...
4,123 ICHI NI SAN Sathorn Soi 1,123 ICHI NI SAN Sathorn Soi 1 123 ICHI NI SAN...
...,...,...
1942,sala ayutthaya staycation (Ayutthaya),sala ayutthaya staycation (Ayutthaya) sala ay...
1943,sala bang pa-in Staycation (Ayutthaya),sala bang pa-in Staycation (Ayutthaya) This st...
1944,อร่อย Together,อร่อย Together Went to sing karaoke and was pl...
1945,中国好味道 China Taste,中国好味道 China Taste Food is relatively less salt...


In [95]:
print("Unique KOL restaurants:", kol_posts['Restaurant Name'].nunique())
print("After merge:", places_merged['Restaurant Name'].nunique())

print("Unique review restaurants:", rest_reviews['input_restaurant_name'].nunique())
print("Final merged:", rest_places_merged['input_restaurant_name'].nunique())

rest_places_merged[rest_places_merged['input_restaurant_name'] == 'JW Cafe at JW Marriott Hotel Bangkok']
#rest_places_merged[rest_places_merged['input_restaurant_name'] == 'Melody Bangkok']
rest[rest['name'] == 'JW Cafe at JW Marriott Hotel Bangkok']

Unique KOL restaurants: 301
After merge: 197
Unique review restaurants: 1947
Final merged: 1947


,restaurant_id,name,days_in_advance


In [96]:
# These are Google Places API tags that leaked into your text column.
# They are NOT customer perception language.
# Remove them before TF-IDF.

redundant_words = [
    'point_of_interest', 'establishment', 'general',
    'food establishment', 'restaurant point_of_interest',
    'food point_of_interest', 'point_of_interest establishment',
    'establishment general', 'restaurant food',
    # Also remove raw Google types that commonly appear:
    'meal_delivery', 'meal_takeaway', 'lodging',
    'tourist_attraction', 'night_club', 'shopping_mall',
    'Bangkok', 'sukhumvit', 'singapore', 'bangkok', 'sukhumvit', 'Singapore'
]

import re

def remove_redundant_words(text):
    """Remove redundant words from review text."""
    cleaned = text
    # Remove multi-word junk phrases first (longer matches first)
    for phrase in sorted(redundant_words, key=len, reverse=True):
        cleaned = cleaned.replace(phrase, ' ')
    # Collapse multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

reviews['text_clean'] = reviews['text'].apply(remove_redundant_words)

In [97]:
# 3. Text vectorization

vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2
)
text_vectors = vectorizer.fit_transform(reviews['text_clean'])

# Verify the junk is gone:
feature_names = vectorizer.get_feature_names_out()
junk_remaining = [f for f in feature_names if 'establishment' in f or 'point_of_interest' in f]
print(f"Junk terms remaining: {junk_remaining}")  # Should be empty

Junk terms remaining: []


In [98]:
"""
Pick K by checking: are all K clusters actually DIFFERENT from each other?
If two clusters have the same distinctive terms, K is too high.
"""
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

feature_names = vectorizer.get_feature_names_out()

for k in [5, 6, 7, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(text_vectors)
    
    print(f"\n{'='*70}")
    print(f"K = {k}")
    print(f"{'='*70}")
    
    cluster_top_terms = {}
    for c in range(k):
        centroid = km.cluster_centers_[c]
        global_avg = km.cluster_centers_.mean(axis=0)
        diff = centroid - global_avg
        top_indices = diff.argsort()[-8:][::-1]
        terms = set(feature_names[i] for i in top_indices)
        cluster_top_terms[c] = terms
        
        mask = labels == c
        print(f"  Cluster {c} ({mask.sum():>4} posts): {', '.join(sorted(terms))}")
    
    # Check overlap: how many clusters share >50% of their top terms?
    duplicates = 0
    for i in range(k):
        for j in range(i+1, k):
            overlap = len(cluster_top_terms[i] & cluster_top_terms[j])
            total = len(cluster_top_terms[i] | cluster_top_terms[j])
            if overlap / total > 0.4:
                duplicates += 1
    
    print(f"\n  → Redundant cluster pairs (>40% term overlap): {duplicates}")
    print(f"  → {'✅ Good — clusters are distinct' if duplicates == 0 else '⚠️ Some clusters are too similar, K may be too high'}")


K = 5
  Cluster 0 ( 118 posts): atmosphere, bistro, cafe, coffee, general, nice, restaurant, shop
  Cluster 1 ( 897 posts): delicious, dishes, food, food delicious, good, price, service, thai
  Cluster 2 ( 168 posts): bar, drinks, great, music, rooftop, sukhumvit, view, wine
  Cluster 3 ( 228 posts): breakfast, hotel, marriott, pattaya, resort, riverside, room, staycation
  Cluster 4 ( 536 posts): central, hua, omakase, phuket, rama, shabu, suki, sushi

  → Redundant cluster pairs (>40% term overlap): 0
  → ✅ Good — clusters are distinct

K = 6
  Cluster 0 ( 120 posts): beach, club, hin, hua, hua hin, pattaya, phuket, resort
  Cluster 1 ( 162 posts): breakfast, floor, hotel, marriott, riverside, room, staycation, sukhumvit
  Cluster 2 (1253 posts): delicious, dishes, food, food delicious, good, service, taste, thai
  Cluster 3 ( 168 posts): bar, great, music, place, rooftop, sukhumvit, view, wine
  Cluster 4 ( 127 posts): buffet, meat, premium, quality, rama, shabu, suki, sushi
  Clus

In [99]:
# Cluster Posts (Initial Clustering at Post Level)
# We still decided to stick to 5 clusters
c = 5
print("\n" + "="*70)
print("STEP 3: Clustering Posts (K-Means with 5 Clusters)")
print("="*70)

kmeans = KMeans(n_clusters=c, random_state=42, n_init=10)
clusters = kmeans.fit_predict(text_vectors)
reviews['cluster'] = clusters

print(f"Assigned {len(reviews)} posts to {c} clusters")
print("\nPost distribution across clusters:")
print(reviews['cluster'].value_counts().sort_index())


STEP 3: Clustering Posts (K-Means with 5 Clusters)
Assigned 1947 posts to 5 clusters

Post distribution across clusters:
cluster
0    118
1    897
2    168
3    228
4    536
Name: count, dtype: int64


In [100]:
# Define Predefined Themes
cluster_themes = {
    0: "Various Menus & Quality Food",
    1: "Buffet and Premium Dining",
    2: "Good location with Great Views",
    3: "Alcohol & rooftop views",
    4: "Fine Cafes & Coffee Culture"
}
# 1) Various Menus & Quality Food
# 2) Buffet and premium dining
# 3) Good location with Great Views
# 4) Alcohol with great view
# 5) Cafes

reviews['theme'] = reviews['cluster'].map(cluster_themes)

In [101]:
reviews.to_csv("reviews.csv")

In [102]:
# Validate Cluster Themes (Check Keywords)
print("\n" + "="*70)
print("STEP 4: Validating Cluster Themes (Auto vs Manual)")
print("="*70)

for cluster_id in range(5):
    cluster_texts = reviews[reviews['cluster'] == cluster_id]['text']
    
    if len(cluster_texts) > 0:
        cluster_vectorizer = TfidfVectorizer(max_features=10, stop_words='english')
        cluster_vectorizer.fit(cluster_texts)
        top_terms = cluster_vectorizer.get_feature_names_out()
        
        print(f"\n Cluster {cluster_id} → {cluster_themes[cluster_id]}")
        print(f"    Posts: {len(cluster_texts)} ({len(cluster_texts)/len(reviews)*100:.1f}%)")
        print(f"    Auto-detected keywords: {', '.join(top_terms)}")
        print(f"    Check if keywords match your theme name!")

# MULTI-THEME RESTAURANT ASSIGNMENT
print("\n" + "="*70)
print("STEP 5: Aggregating to Restaurant Level (Multi-Theme Method)")
print("="*70)

# Count posts per restaurant per cluster
restaurant_theme_counts = (
    reviews.groupby(['restaurant name', 'cluster', 'theme'])
    .size()
    .reset_index(name='post_count')
)

# Get total posts per restaurant
restaurant_post_totals = reviews.groupby('restaurant name').size().reset_index(name='total_posts')

# Merge to calculate percentages
restaurant_theme_counts = restaurant_theme_counts.merge(
    restaurant_post_totals, 
    on='restaurant name'
)

restaurant_theme_counts['percentage'] = ( # Calculate percentage of posts for each theme per restaurant
    restaurant_theme_counts['post_count'] / restaurant_theme_counts['total_posts'] * 100
)

print(f"\n Analyzing {len(restaurant_post_totals)} unique restaurants...")


STEP 4: Validating Cluster Themes (Auto vs Manual)

 Cluster 0 → Various Menus & Quality Food
    Posts: 118 (6.1%)
    Auto-detected keywords: atmosphere, bangkok, cafe, delicious, food, good, great, restaurant, service, staff
    Check if keywords match your theme name!

 Cluster 1 → Buffet and Premium Dining
    Posts: 897 (46.1%)
    Auto-detected keywords: atmosphere, bangkok, delicious, food, good, great, restaurant, service, staff, thai
    Check if keywords match your theme name!

 Cluster 2 → Good location with Great Views
    Posts: 168 (8.6%)
    Auto-detected keywords: bangkok, bar, establishment, food, good, great, restaurant, rooftop, service, staff
    Check if keywords match your theme name!

 Cluster 3 → Alcohol & rooftop views
    Posts: 228 (11.7%)
    Auto-detected keywords: bangkok, delicious, food, good, great, hotel, pattaya, restaurant, service, staff
    Check if keywords match your theme name!

 Cluster 4 → Fine Cafes & Coffee Culture
    Posts: 536 (27.5%)
 

In [103]:
# Keep themes with at least 20% of restaurant's posts
THRESHOLD = 20  # Adjust this threshold (e.g., 15, 20, 25, 30)

significant_themes = restaurant_theme_counts[
    restaurant_theme_counts['percentage'] >= THRESHOLD
].copy()

print(f"\n Keeping themes that represent ≥{THRESHOLD}% of a restaurant's posts")
print(f"   Found {len(significant_themes)} restaurant-theme combinations")

# Create multi-theme labels
restaurant_multi_themes = (
    significant_themes.groupby('restaurant name')
    .agg({
        'theme': lambda x: ' + '.join(sorted(x)),  # Combine themes
        'cluster': lambda x: list(x),  # Keep cluster IDs
        'percentage': lambda x: list(x)  # Keep percentages
    })
    .reset_index()
)

restaurant_multi_themes.columns = ['restaurant name', 'combined_themes', 'clusters', 'percentages']

print("\n" + "="*70)
print("RESTAURANT THEME DISTRIBUTION (Multi-Theme Assignment)")
print("="*70)
print(restaurant_multi_themes['combined_themes'].value_counts().head(15))

# Also create primary theme (for visualization)
# Assign each restaurant to its DOMINANT theme (highest percentage)
restaurant_primary_theme = (
    restaurant_theme_counts
    .sort_values('percentage', ascending=False)
    .groupby('restaurant name')
    .first()
    .reset_index()
)

restaurant_primary_theme = restaurant_primary_theme[[
    'restaurant name', 'cluster', 'theme', 'percentage', 'post_count', 'total_posts'
]]

print("\n" + "="*70)
print("PRIMARY THEME DISTRIBUTION (Dominant Theme per Restaurant)")
print("="*70)
print(restaurant_primary_theme['theme'].value_counts())


 Keeping themes that represent ≥20% of a restaurant's posts
   Found 1947 restaurant-theme combinations

RESTAURANT THEME DISTRIBUTION (Multi-Theme Assignment)
combined_themes
Buffet and Premium Dining         897
Fine Cafes & Coffee Culture       536
Alcohol & rooftop views           228
Good location with Great Views    168
Various Menus & Quality Food      118
Name: count, dtype: int64

PRIMARY THEME DISTRIBUTION (Dominant Theme per Restaurant)
theme
Buffet and Premium Dining         897
Fine Cafes & Coffee Culture       536
Alcohol & rooftop views           228
Good location with Great Views    168
Various Menus & Quality Food      118
Name: count, dtype: int64


In [104]:
# STEP 7: Create Restaurant-Level Embeddings for Visualization
print("\n" + "="*70)
print("STEP 6: Creating Restaurant-Level Embeddings")
print("="*70)

restaurant_embeddings = []
restaurant_names_list = []

# Reset index to ensure alignment with text_vectors
reviews_reset = reviews.reset_index(drop=True)

for rest_id in restaurant_primary_theme['restaurant name']:
    # Get positions (not indices) of posts for this restaurant
    post_positions = reviews_reset[reviews_reset['restaurant name'] == rest_id].index.tolist()
    
    if len(post_positions) == 0:
        continue
    
    # Average their TF-IDF vectors using positions
    rest_vector = text_vectors[post_positions].mean(axis=0)
    restaurant_embeddings.append(np.array(rest_vector).flatten())
    restaurant_names_list.append(rest_id)

restaurant_embeddings = np.array(restaurant_embeddings)
print(f" Created embeddings for {len(restaurant_embeddings)} restaurants")


STEP 6: Creating Restaurant-Level Embeddings


 Created embeddings for 1947 restaurants


In [105]:
# STEP 8: Dimensionality Reduction (200D → 2D using UMAP)
print("\n" + "="*70)
print("STEP 7: Reducing Dimensions (200D → 2D for Visualization)")
print("="*70)

# Can experiment with UMAP parameters like n_neighbors and min_dist to see how it affects the clustering in 2D space
reducer = UMAP(
    n_components=2,
    n_neighbors=30,    # ← Increase from 15 (preserves more global structure)
    min_dist=1,      # ← Increase from 0.1 (allows more spread)
    metric='cosine',   # ← Use cosine similarity (better for text)
    random_state=42
)
embeddings_2d = reducer.fit_transform(restaurant_embeddings)
print(f" Reduced to 2D coordinates")


STEP 7: Reducing Dimensions (200D → 2D for Visualization)


/Users/jadentyh/Desktop/IS455/Project/OPE/venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



 Reduced to 2D coordinates


In [106]:
# viz_df.to_csv("viz_df.csv")

In [107]:
# Create Visualization DataFrame
viz_df = pd.DataFrame({
    'restaurant name': restaurant_names_list, # Align restaurant names with the order of embeddings
    'UMAP Component 1 (Semantic Similarity Axis)': embeddings_2d[:, 0], # UMAP component 1 captures the main semantic differences between restaurants based on their reviews and google text
    'UMAP Component 2 (Theme Variation Axis)': embeddings_2d[:, 1], # UMAP component 2 captures the variation in themes among restaurants
})

# Merge with primary theme (for color)
viz_df = viz_df.merge( # Merge primary theme info for coloring the points in the visualization
    restaurant_primary_theme[['restaurant name', 'cluster', 'theme', 'total_posts', 'percentage']], 
    on='restaurant name'
)

# Merge with multi-themes (for hover info)
viz_df = viz_df.merge(
    restaurant_multi_themes[['restaurant name', 'combined_themes']], 
    on='restaurant name',
    how='left'
)

# Fill single-theme restaurants
viz_df['combined_themes'] = viz_df['combined_themes'].fillna(viz_df['theme'])

# Rename for clarity
viz_df = viz_df.rename(columns={
    'theme': 'Primary Theme',
    'total_posts': 'Number of Posts',
    'percentage': 'Primary Theme %',
    'combined_themes': 'All Themes'
})

print(f"\n Prepared {len(viz_df)} restaurants for visualization")


 Prepared 1947 restaurants for visualization


In [108]:
# Create Interactive Plotly Visualization
print("\n" + "="*70)
print("STEP 8: Creating Interactive Visualization")
print("="*70)

fig = px.scatter( # Create an interactive scatter plot with Plotly Express
    viz_df,
    x='UMAP Component 1 (Semantic Similarity Axis)',
    y='UMAP Component 2 (Theme Variation Axis)',
    color='Primary Theme',
    #size='Number of Posts',  # Bigger dots = more posts about this restaurant # Comment out if you want uniform dot sizes
    hover_data={ # Show these details when hovering over a restaurant
        'restaurant name': True,
        'Primary Theme': True,
        'All Themes': True,  # Shows multi-theme assignments
        'Number of Posts': True,
        'Primary Theme %': ':.1f',
        'UMAP Component 1 (Semantic Similarity Axis)': ':.2f',
        'UMAP Component 2 (Theme Variation Axis)': ':.2f'
    },
    title='<b>Restaurant Segmentation by Customer Perception Themes</b><br>' + 
          '<sub>(1 Dot = 1 Restaurant | Size = Number of Posts | Color = Primary Theme)</sub>',
    color_discrete_sequence=px.colors.qualitative.Set3,
)

fig.update_layout( # Customize layout for better readability and aesthetics
    font=dict(size=12),
    title_font=dict(size=16),
    width=1400,
    height=800,
    legend=dict(
        title="Primary Customer Perception Theme",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    hovermode='closest'
)

# Update marker size range for better visibility
fig.update_traces(
    marker=dict(
        sizemode='diameter',
        sizemin=5,
        sizeref=2.*max(viz_df['Number of Posts'])/(40.**2),
        line=dict(width=1, color='DarkSlateGrey')
    )
)

# Add cluster centroids with labels
centroids = viz_df.groupby("Primary Theme")[
    ['UMAP Component 1 (Semantic Similarity Axis)', 
     'UMAP Component 2 (Theme Variation Axis)']
].mean().reset_index()

for _, row in centroids.iterrows():
    fig.add_annotation(
        x=row['UMAP Component 1 (Semantic Similarity Axis)'],
        y=row['UMAP Component 2 (Theme Variation Axis)'],
        text=f"<b>{row['Primary Theme']}</b>",
        showarrow=False,
        font=dict(size=10, color="black"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="black",
        borderwidth=1.5,
        borderpad=5
    )

fig.write_html("restaurant_clusters_multi_theme.html")
print("\n Visualization saved to 'restaurant_clusters_multi_theme.html'")


STEP 8: Creating Interactive Visualization

 Visualization saved to 'restaurant_clusters_multi_theme.html'


In [109]:
# Save Results to CSV
print("\n" + "="*70)
print("STEP 9: Saving Results")
print("="*70)

# Save restaurant-level data with all themes
output_df = viz_df[[
    'restaurant name', 
    'Primary Theme', 
    'All Themes', 
    'Number of Posts',
    'Primary Theme %'
]].copy()

output_df.to_csv('restaurants_with_multi_themes.csv', index=False)
print(" Saved 'restaurants_with_multi_themes.csv'")

# Also save detailed theme breakdown
restaurant_theme_counts.to_csv('restaurant_theme_details.csv', index=False)
print(" Saved 'restaurant_theme_details.csv' (shows % for each theme)")


STEP 9: Saving Results
 Saved 'restaurants_with_multi_themes.csv'
 Saved 'restaurant_theme_details.csv' (shows % for each theme)


In [110]:
# Summary Statistics
print("\n" + "="*70)
print(" SUMMARY STATISTICS")
print("="*70)

print(f"\n Total Restaurants: {len(viz_df)}")
print(f" Total Posts: {viz_df['Number of Posts'].sum()}")
print(f" Avg Posts per Restaurant: {viz_df['Number of Posts'].mean():.1f}")

print("\n Single-Theme vs Multi-Theme Restaurants:")
single_theme = viz_df[viz_df['All Themes'] == viz_df['Primary Theme']]
multi_theme = viz_df[viz_df['All Themes'] != viz_df['Primary Theme']]

print(f"   Single-theme restaurants: {len(single_theme)} ({len(single_theme)/len(viz_df)*100:.1f}%)")
print(f"   Multi-theme restaurants: {len(multi_theme)} ({len(multi_theme)/len(viz_df)*100:.1f}%)")

print("\n Most Common Multi-Theme Combinations:")
if len(multi_theme) > 0:
    print(multi_theme['All Themes'].value_counts().head(10))

print("\n CLUSTERING COMPLETE with new data!")
print("="*70)


 SUMMARY STATISTICS

 Total Restaurants: 1947
 Total Posts: 1947
 Avg Posts per Restaurant: 1.0

 Single-Theme vs Multi-Theme Restaurants:
   Single-theme restaurants: 1947 (100.0%)
   Multi-theme restaurants: 0 (0.0%)

 Most Common Multi-Theme Combinations:

 CLUSTERING COMPLETE with new data!


In [111]:
# Merge with restaurant to get restaurant id
clustering_results = pd.merge(
    output_df,
    rest,
    left_on="restaurant name",
    right_on="name",
    how="left"
)

viz_results = pd.merge(
    viz_df,
    rest,
    left_on="restaurant name",
    right_on="name",
    how="left"
)

clustering_results.to_csv('clustering_results.csv', index=False)

In [112]:
clustering_results.shape

(1947, 8)

In [113]:
clustering_results['restaurant_id'].nunique()
# clustering_results['restaurant name'].nunique() 1947

1732

In [ ]:
viz_df['cluster'].isnull().sum()
viz_df[viz_df['restaurant name']=='Myste']
viz_results.head() # Shape = 1947 rows, 10 columns

,restaurant name,UMAP Component 1 (Semantic Similarity Axis),UMAP Component 2 (Theme Variation Axis),cluster,Primary Theme,Number of Posts,Primary Theme %,All Themes,restaurant_id,name,days_in_advance
0,&T modern Thai grill,9.959900,0.326509,1,Buffet and Premium Dining,1,100.0,Buffet and Premium Dining,NaN,<NA>,NaN
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,8.217741,-4.865994,4,Fine Cafes & Coffee Culture,1,100.0,Fine Cafes & Coffee Culture,3379.0,'@RICE Restaurant by At Rice Resort (Nakhon Na...,90.0
2,100 Degrees Hotpot Central Chaengwattana,7.443743,1.279914,4,Fine Cafes & Coffee Culture,1,100.0,Fine Cafes & Coffee Culture,6420.0,100 Degrees Hotpot Central Chaengwattana,90.0
3,100 Degrees Hotpot Cosmo Bazaar,10.413542,-3.042263,1,Buffet and Premium Dining,1,100.0,Buffet and Premium Dining,6421.0,100 Degrees Hotpot Cosmo Bazaar,90.0
4,123 ICHI NI SAN Sathorn Soi 1,13.322515,-3.879483,1,Buffet and Premium Dining,1,100.0,Buffet and Premium Dining,2724.0,123 ICHI NI SAN Sathorn Soi 1,90.0


In [ ]:
# STEP 10: Export dashboard artifacts for Streamlit (UPDATED)
# - Ensures a cluster map is available on the frontend by exporting x/y coordinates.
# - Fixes column naming so loader.py can always find: restaurant_id, name, cluster_id, cluster_label, x, y
# - Uses viz_results (UMAP restaurant-level map) as the primary source for x/y.
# - Falls back gracefully if some fields are missing.

from pathlib import Path
import numpy as np
import pandas as pd

print("\n" + "="*70)
print("STEP 10: Export dashboard artifacts for Streamlit (UPDATED)")
print("="*70)

def _find_project_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "frontend_dashboard").exists():
            return path
    return Path.cwd()

project_root = _find_project_root()
export_dir = project_root / "frontend_dashboard" / "data" / "clustering"
export_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) restaurant_cluster_assignments.parquet
#    (THIS is what powers the cluster map)
# -----------------------------
assignments = pd.DataFrame()

# Prefer viz_results because it contains UMAP x/y
if "viz_results" in globals() and isinstance(viz_results, pd.DataFrame) and len(viz_results):
    keep_base = ["restaurant name", "cluster", "Primary Theme", "UMAP Component 1 (Semantic Similarity Axis)", "UMAP Component 2 (Theme Variation Axis)"]
    assignments = viz_results[keep_base].copy()
elif "clustering_results" in globals() and isinstance(clustering_results, pd.DataFrame) and len(clustering_results):
    assignments = clustering_results.copy()
elif Path("clustering_results.csv").exists():
    assignments = pd.read_csv("clustering_results.csv")

# # --- Require clustering_results for the cluster map ---
# if "clustering_results" not in globals() or not isinstance(clustering_results, pd.DataFrame) or clustering_results.empty:
#     raise RuntimeError("clustering_results is missing/empty. Run the clustering cells before exporting.")

# assignments = clustering_results.copy()

if assignments.empty:
    raise RuntimeError(
        "No clustering dataframe found. Expected viz_results or clustering_results to exist "
        "before exporting artifacts."
    )

#  df
'''
restaurant name', 'UMAP Component 1 (Semantic Similarity Axis)',
       'UMAP Component 2 (Theme Variation Axis)', 'cluster', 'Primary Theme',
       'Number of Posts', 'Primary Theme %', 'All Themes'],
      dtype='str')
      '''

# Standardize columns
rename_map = {
    "restaurant name": "name",
    "UMAP Component 1 (Semantic Similarity Axis)": "x",
    "UMAP Component 2 (Theme Variation Axis)": "y",
    'cluster': 'cluster_id',
    'Primary Theme': 'cluster_label',
    # Allow already-standard columns to pass through
}

assignments = assignments.rename(columns={k: v for k, v in rename_map.items() if k in assignments.columns})

# Keep ONLY the columns the dashboard needs (prevents duplicate-column explosions)
keep_base = ["name", "cluster_id", "cluster_label", "x", "y"]
missing = [c for c in keep_base if c not in assignments.columns]
if missing:
    raise RuntimeError(f"Missing required columns in viz_df/viz_results: {missing}")

# Ensure required columns exist
if "name" not in assignments.columns:
    # Try alternative source column
    if "restaurant name" in assignments.columns:
        assignments["name"] = assignments["restaurant name"]
    else:
        raise RuntimeError("Assignments missing restaurant name column (expected 'name' or 'restaurant name').")

if "cluster_id" not in assignments.columns:
    # If cluster id isn't present but label exists, create a dummy cluster id
    assignments["cluster_id"] = pd.factorize(assignments.get("cluster_label", assignments["name"]))[0]

if "cluster_label" not in assignments.columns:
    assignments["cluster_label"] = assignments["cluster_id"].apply(lambda v: f"Cluster {v}")

# Ensure x/y exist (cluster map requires them)
if "x" not in assignments.columns or "y" not in assignments.columns:
    # If x/y are missing, create a simple placeholder embedding to avoid breaking the dashboard
    # (Better than failing; but you should still export viz_results next run.)
    rng = np.random.default_rng(42)
    if "x" not in assignments.columns:
        assignments["x"] = rng.normal(loc=0.0, scale=1.0, size=len(assignments))
    if "y" not in assignments.columns:
        assignments["y"] = rng.normal(loc=0.0, scale=1.0, size=len(assignments))
    print("⚠️  x/y UMAP coordinates missing. Exported placeholder coordinates. "
          "To fix properly, ensure viz_results is created before this cell runs.")

# Attach restaurant_id if possible (needed for joining to other modules)
# Prefer clustering_results merge output if it exists; else try to merge with rest dataframe.
if "restaurant_id" not in assignments.columns:
    if "rest" in globals() and isinstance(rest, pd.DataFrame) and "name" in rest.columns and "restaurant_id" in rest.columns:
        tmp = rest[["restaurant_id", "name"]].drop_duplicates("name")
        assignments = assignments.merge(tmp, on="name", how="left")
    else:
        assignments["restaurant_id"] = np.nan

# Normalize types
# Check if cluster_id contains strings that aren't purely numeric
if assignments["cluster_id"].dtype == object:
    # Try to extract just the numbers (e.g., "Cluster 3" -> 3)
    extracted = assignments["cluster_id"].astype(str).str.extract(r'(\d+)')[0]
    
    # If extraction failed (e.g., they were labeled "A", "B"), use factorize
    if extracted.isnull().all():
        assignments["cluster_id"] = pd.factorize(assignments["cluster_id"])[0]
    else:
        assignments["cluster_id"] = extracted

# Now safely convert to int
assignments["cluster_id"] = pd.to_numeric(assignments["cluster_id"], errors="coerce").fillna(-1).astype(int)
assignments["restaurant_id"] = pd.to_numeric(assignments["restaurant_id"], errors="coerce")

# # Keep only what the dashboard needs
# assignments_out = assignments[[
#     c for c in ["restaurant_id", "name", "cluster_id", "cluster_label", "x", "y"]
#     if c in assignments.columns
# ]].copy()
print(f"Assignments has {assignments['cluster_id'].isnull().sum()} null cluster values")
assignments_out = assignments[[
    "restaurant_id","name","cluster_id","cluster_label","x","y"
]].copy()

# Add optional fields expected by some loaders/pages
assignments_out["cluster_confidence"] = np.nan
assignments_out["year_month"] = pd.NaT

print(f"Assignment out has {assignments_out['cluster_label'].isnull().sum()} null cluster values")
assignments_path = export_dir / "restaurant_cluster_assignments.parquet"
assignments_out.to_parquet(assignments_path, index=False)
print(f"Saved: {assignments_path} ({len(assignments_out):,} rows)")

# -----------------------------
# 2) cluster_keywords.parquet
# -----------------------------
# Build keywords per cluster from TF-IDF centroids (more meaningful than using theme_counts)
keywords_rows = []
if "kmeans" in globals() and "vectorizer" in globals():
    feature_names = vectorizer.get_feature_names_out()
    centroids = kmeans.cluster_centers_
    k = centroids.shape[0]
    for cid in range(k):
        top_idx = np.argsort(centroids[cid])[-50:][::-1]
        for rank, idx in enumerate(top_idx, start=1):
            keywords_rows.append({
                "cluster_id": int(cid),
                "keyword": str(feature_names[idx]),
                "weight": float(centroids[cid][idx]),
                "rank": int(rank),
            })
else:
    # Fallback: token frequency from raw text if kmeans/vectorizer not present
    tmp_reviews = reviews.copy() if "reviews" in globals() else pd.DataFrame()
    if not tmp_reviews.empty and "cluster" in tmp_reviews.columns and "text" in tmp_reviews.columns:
        tmp_reviews = tmp_reviews.rename(columns={"cluster": "cluster_id", "text": "raw_text"})
        for cid, cdf in tmp_reviews.groupby("cluster_id"):
            tokens = (
                cdf["raw_text"].fillna("").astype(str)
                .str.lower()
                .str.findall(r"[a-zA-Z]{3,}")
                .explode()
                .dropna()
            )
            top_terms = tokens.value_counts().head(50)
            for rank, (word, weight) in enumerate(top_terms.items(), start=1):
                keywords_rows.append({"cluster_id": int(cid), "keyword": word, "weight": float(weight), "rank": int(rank)})

keywords = pd.DataFrame(keywords_rows, columns=["cluster_id", "keyword", "weight", "rank"])
keywords_path = export_dir / "cluster_keywords.parquet"
keywords.to_parquet(keywords_path, index=False)
print(f"Saved: {keywords_path} ({len(keywords):,} rows)")

# -----------------------------
# 3) restaurant_text_corpus.parquet
# -----------------------------
# Keep per-restaurant text corpus + cluster assignment (useful for search/debug)
if "reviews" in globals() and isinstance(reviews, pd.DataFrame) and len(reviews):
    text_corpus = reviews.copy()
else:
    text_corpus = pd.DataFrame(columns=["restaurant name", "text", "cluster"])

text_corpus = text_corpus.rename(columns={
    "restaurant name": "name",
    "text": "raw_text",
    "text_clean": "clean_text",
    "cluster": "cluster_id",
})
if "text_id" not in text_corpus.columns:
    text_corpus["text_id"] = np.arange(1, len(text_corpus) + 1)

# Ensure required fields
if "clean_text" not in text_corpus.columns:
    text_corpus["clean_text"] = (
        text_corpus["raw_text"].fillna("").astype(str)
        .str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    )
if "year_month" not in text_corpus.columns:
    text_corpus["year_month"] = pd.NaT

text_cols = [c for c in ["name", "text_id", "raw_text", "clean_text", "cluster_id", "year_month"] if c in text_corpus.columns]
text_corpus = text_corpus[text_cols].copy()

text_path = export_dir / "restaurant_text_corpus.parquet"
text_corpus.to_parquet(text_path, index=False)
print(f"Saved: {text_path} ({len(text_corpus):,} rows)")

# -----------------------------
# 4) cluster_strategy_outcomes.parquet
# -----------------------------
# Optional: only if marketing uplift file exists
marketing_path = project_root / "_3_marketing" / "activity_performance_with_roi.csv"
if marketing_path.exists() and assignments_out["restaurant_id"].notna().any():
    marketing = pd.read_csv(marketing_path)
    marketing["restaurant_id"] = pd.to_numeric(marketing.get("restaurant_id"), errors="coerce")

    assign_join = assignments_out[["restaurant_id", "name", "cluster_id", "cluster_label"]].dropna(subset=["restaurant_id"]).copy()
    assign_join["restaurant_id"] = assign_join["restaurant_id"].astype(int)

    strategy = marketing.merge(assign_join, on="restaurant_id", how="inner")

    # Basic computed fields used by downstream strategy logic
    strategy["roi"] = pd.to_numeric(strategy.get("roi"), errors="coerce")
    strategy["applied_date"] = pd.to_datetime(strategy.get("activity_start"), errors="coerce")
    strategy["sample_size"] = 1

    strategy_path = export_dir / "cluster_strategy_outcomes.parquet"
    strategy.to_parquet(strategy_path, index=False)
    print(f"Saved: {strategy_path} ({len(strategy):,} rows)")
else:
    strategy_path = export_dir / "cluster_strategy_outcomes.parquet"
    pd.DataFrame().to_parquet(strategy_path, index=False)
    print(f"Saved: {strategy_path} (0 rows; marketing file missing or no restaurant_id mapping)")


STEP 10: Export dashboard artifacts for Streamlit (UPDATED)
Assignments has 0 null cluster values
Assignment out has 0 null cluster values
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/restaurant_cluster_assignments.parquet (1,947 rows)
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/cluster_keywords.parquet (250 rows)
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/restaurant_text_corpus.parquet (1,947 rows)
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/cluster_strategy_outcomes.parquet (1,101 rows)


In [124]:
import pandas as pd

assign = pd.read_parquet("../frontend_dashboard/data/clustering/restaurant_cluster_assignments.parquet")
uncl = assign[assign["cluster_id"] == -1].copy()

print("Unclustered rows:", len(uncl))
print(uncl[["restaurant_id", "name", "cluster_label"]].head(20))

Unclustered rows: 0
Empty DataFrame
Columns: [restaurant_id, name, cluster_label]
Index: []
